In [ ]:
# 02_cleaning_eda.ipynb — Cell A: inspect raw CSVs before schema design
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / "data" / "raw"

files = [
    "entsoe_generation_AT.csv",
    "entsoe_load_AT.csv",
    "entsoe_prices_AT.csv",
    "weather_vienna.csv",
]

for fname in files:
    path = RAW / fname
    if not path.exists():
        print(f"!! MISSING: {path}\n")
        continue
    size_kb = path.stat().st_size / 1024
    print("=" * 80)
    print(f"  {fname}   ({size_kb:,.0f} KB)")
    print("=" * 80)
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            print(f"L{i}: {line.rstrip()[:160]}")
    print()

In [ ]:
# 02_cleaning_eda.ipynb — Cell B: open DuckDB + create schema
import duckdb
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

DB_PATH = PROCESSED / "austria_energy.duckdb"
con = duckdb.connect(str(DB_PATH))

# Clean slate — drop everything for safe re-runs while iterating
for tbl in ["generation_15min", "demand_15min",
            "generation", "demand", "prices", "weather", "owid_energy_at"]:
    con.execute(f"DROP TABLE IF EXISTS {tbl}")

## This version is just for DuckDB/PostgreSQL dialect
## For example, for generation_15min, the following is the correct syntax:

## con.execute("""
## CREATE OR REPLACE TABLE generation_15min (
##     ts_utc      TIMESTAMPTZ NOT NULL,
##     fuel_type   VARCHAR     NOT NULL,
##     flow        VARCHAR     NOT NULL,
##     mw          DOUBLE,
##     PRIMARY KEY (ts_utc, fuel_type, flow)
## )
## """)

# ── Staging (15-min, long format) ────────────────────────────────────────
con.execute("""
CREATE TABLE generation_15min (
    ts_utc      TIMESTAMPTZ NOT NULL,
    fuel_type   VARCHAR     NOT NULL,
    flow        VARCHAR     NOT NULL,      -- 'generation' or 'consumption'
    mw          DOUBLE,
    PRIMARY KEY (ts_utc, fuel_type, flow)
)
""")

con.execute("""
CREATE TABLE demand_15min (
    ts_utc     TIMESTAMPTZ PRIMARY KEY,
    demand_mw  DOUBLE          -- allow NULL; we'll count gaps in QA
)
""")

# ── Final hourly tables (no resampling needed — native hourly) ──────────
con.execute("""
CREATE TABLE prices (
    ts_utc         TIMESTAMPTZ PRIMARY KEY,
    price_eur_mwh  DOUBLE       -- allow NULL; we'll count gaps in QA
)
""")

con.execute("""
CREATE TABLE weather (
    ts_utc          TIMESTAMPTZ PRIMARY KEY,
    temperature_c   DOUBLE,
    solar_wm2       DOUBLE,
    wind_kmh        DOUBLE,
    precip_mm       DOUBLE,
    cloudcover_pct  DOUBLE
)
""")

# generation, demand, owid_energy_at are created via CTAS in later cells.

print(con.sql("SHOW TABLES").df())

In [ ]:
# 02_cleaning_eda.ipynb — Cell C: load OWID + prices + weather
import pandas as pd

RAW = PROJECT_ROOT / "data" / "raw"

# ── 1. OWID — annual, no TZ, CTAS straight from CSV ─────────────────────
con.execute(f"""
CREATE TABLE owid_energy_at AS
SELECT *
FROM read_csv_auto('{RAW / "owid_energy_AUT.csv"}', header=true)
""")
n = con.sql("SELECT COUNT(*) FROM owid_energy_at").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"owid_energy_at: {n} rows")

# ── 2. Prices — timestamps already carry offsets, parse to UTC ──────────
df_p = pd.read_csv(RAW / "entsoe_prices_AT.csv", index_col=0, parse_dates=True)
df_p.index = pd.to_datetime(df_p.index, utc=True)         # to UTC
df_p = df_p.rename_axis("ts_utc").reset_index()
df_p.columns = ["ts_utc", "price_eur_mwh"]
# there exists an explicit form of insert
# It is for assuring to be name sensitive about columns and not just order
con.execute("INSERT INTO prices SELECT * FROM df_p")
n = con.sql("SELECT COUNT(*) FROM prices").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"prices: {n} rows")

# ── 3. Weather — TZ-naive, localize to Vienna then to UTC ───────────────
# df_w = pd.read_csv(RAW / "weather_vienna.csv",
#                    index_col="timestamp", parse_dates=True)
# df_w.index = (
#     df_w.index
#         .tz_localize("Europe/Vienna",
#                      ambiguous="infer",          # handles DST fall-back
#                      nonexistent="shift_forward")# handles DST spring-forward
#         .tz_convert("UTC")
# )
# df_w = df_w.rename(columns={
#     "temperature_2m":      "temperature_c",
#     "shortwave_radiation": "solar_wm2",
#     "windspeed_10m":       "wind_kmh",
#     "precipitation":       "precip_mm",
#     "cloudcover":          "cloudcover_pct",
# }).rename_axis("ts_utc").reset_index()
# con.execute("INSERT INTO weather SELECT * FROM df_w")
# n = con.sql("SELECT COUNT(*) FROM weather").fetchone()[0]
# print(f"weather: {n} rows")

In [ ]:
# TODO: Run this after you have the UTC version

# ── 3. Weather — already UTC, just tag the index ────────────────────────
df_w = pd.read_csv(RAW / "weather_vienna.csv",
                   index_col="timestamp", parse_dates=True)
df_w.index = df_w.index.tz_localize("UTC")     # naive → UTC label, no shift  # pyright: ignore[reportAttributeAccessIssue]
df_w = df_w.rename(columns={
    "temperature_2m":      "temperature_c",
    "shortwave_radiation": "solar_wm2",
    "windspeed_10m":       "wind_kmh",
    "precipitation":       "precip_mm",
    "cloudcover":          "cloudcover_pct",
}).rename_axis("ts_utc").reset_index()
con.execute("INSERT INTO weather SELECT * FROM df_w")
n = con.sql("SELECT COUNT(*) FROM weather").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"weather: {n} rows")

In [ ]:
df_w.tail()

In [ ]:
#! just till UTC version is here 

# ── 3. Weather — TZ-naive (Europe/Vienna local), workaround until UTC refetch ─
# Loader now patched to request UTC, but Open-Meteo archive is 504-ing today.
# Two DST conventions, both documented:
#
#  ambiguous=False
#    On fall-back days the file has ONE row labelled 02:00 (the duplicate is
#    collapsed by Open-Meteo). We tag it as CET (+01:00) — the second/post-
#    fallback occurrence. Consequence: on each fall-back day the 00:00 UTC
#    slot has no weather row (~6 rows out of ~52,000 across 2019–2024).
#
#  nonexistent="shift_forward"
#    On spring-forward days the file may contain a 02:00 row that didn't
#    actually exist in local time. We shift it to 03:00 — which can create a
#    duplicate UTC hour with the real 03:00 row, dropped below with a log.

df_w = pd.read_csv(RAW / "weather_vienna.csv",
                   index_col="timestamp", parse_dates=True)

df_w.index = (
    df_w.index
        .tz_localize("Europe/Vienna",  # pyright: ignore[reportAttributeAccessIssue]
                     ambiguous=False,
                     nonexistent="shift_forward")
        .tz_convert("UTC")
)

# Catch any duplicates the spring-forward shift may have produced
n_dup = df_w.index.duplicated().sum()
if n_dup:
    print(f"  dropped {n_dup} duplicate UTC hours from DST spring-forward")
    df_w = df_w[~df_w.index.duplicated(keep="first")]

df_w = df_w.rename(columns={
    "temperature_2m":      "temperature_c",
    "shortwave_radiation": "solar_wm2",
    "windspeed_10m":       "wind_kmh",
    "precipitation":       "precip_mm",
    "cloudcover":          "cloudcover_pct",
}).rename_axis("ts_utc").reset_index()

con.execute("""
    INSERT INTO weather (ts_utc, temperature_c, solar_wm2,
                         wind_kmh, precip_mm, cloudcover_pct)
    SELECT ts_utc, temperature_c, solar_wm2,
           wind_kmh, precip_mm, cloudcover_pct
    FROM df_w
""")
n = con.sql("SELECT COUNT(*) FROM weather").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"weather: {n} rows")

In [ ]:
# 02_cleaning_eda.ipynb — Cell D: sanity check what's in the DB
con.sql("""
SELECT 'prices'  AS tbl, COUNT(*) AS rows,
       MIN(ts_utc) AS first_ts, MAX(ts_utc) AS last_ts
FROM prices
UNION ALL
SELECT 'weather', COUNT(*), MIN(ts_utc), MAX(ts_utc) FROM weather
UNION ALL
SELECT 'owid',    COUNT(*), NULL, NULL FROM owid_energy_at
""").df()

In [ ]:
con.sql("""
SELECT 'demand_15min' AS tbl,
       COUNT(*)                              AS rows,
       COUNT(*) FILTER (WHERE demand_mw IS NULL) AS nulls,
       ROUND(100.0 * COUNT(*) FILTER (WHERE demand_mw IS NULL) / COUNT(*), 3)
                                             AS pct_null
FROM demand_15min
UNION ALL
SELECT 'generation_15min',
       COUNT(*),
       COUNT(*) FILTER (WHERE mw IS NULL),
       ROUND(100.0 * COUNT(*) FILTER (WHERE mw IS NULL) / COUNT(*), 3)
FROM generation_15min
""").df()